# Baseline — the released MedShapeNet weights

The comparison this project reports against: the original authors' published
weights, run on this project's aligned data. It answers *"how much does training
on skulls buy you over a general foundation model?"*

**The numbers are not computed here.** `src/eval/eval_pretrained_baseline.py`
produces them — five stochastic draws per skull, once per fold — and this
notebook reads the frozen result. What it does do is build the model, verify the
weights actually loaded, and show one skull.

### Why this exists rather than the upstream inference notebook

That notebook was the only way to run inference in July, before this project had
a data pipeline. Three problems made its numbers unusable:

1. **Alignment was re-broken.** It normalised the partial cloud and the ground
   truth *separately*, each with its own centroid and radius. Measured on skull
   000: the ground-truth-to-input nearest distance inflated **1.31x**.
2. **No `scale_mm`**, so the metrics are dimensionless and cannot be compared
   with anything reported in millimetres.
3. It scored the **first 50 of 100** skulls, not the validation set.

Note the first problem came from wiring this project's data into that notebook,
not from upstream.

## 1 · Setup

In [1]:
import os
import sys

REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
assert os.path.isdir(os.path.join(REPO, "src", "models")), f"repo root not found from {os.getcwd()}"
for sub in ("src/models", "src/eval", "src/data"):
    sys.path.insert(0, os.path.join(REPO, sub))

os.environ.setdefault("HF_HOME", "/root/.cache/huggingface")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import glob
import numpy as np
import pandas as pd
import paths

FOLD = "cd_rep05_full_f0"       # any config's fold 0 -- all four validate the same skulls
DEVICE = "/GPU:0"               # "/CPU:0" if the 296.9M model (BERT included) will not fit

cache = np.load(os.path.join(REPO, paths.DATA_CACHE))
ids, inputs, gt, scale_mm = cache["ids"], cache["inputs"], cache["gt"], cache["scale_mm"]
print(f"cache: {len(ids)} pairs, inputs {inputs.shape}, gt {gt.shape}")

cache: 100 pairs, inputs (100, 4096, 3), gt (100, 6144, 3)


## 2 · Load the released weights

⚠️ **They only fit `src/models/msn_demo_arch.py`**, the upstream architecture
lifted verbatim — never `msn_skullfix.paper()`.

The failure mode is silent, which is why the check below exists.
`load_weights(by_name=True, skip_mismatch=True)` returns without raising while
matching 3 of 32 weight groups, leaving ~96% of the network randomly initialised.
The cause is layer nesting, not shape: upstream wraps Dense+ReLU in a nested
model, so the checkpoint stores `E-IN_LBR1/E-IN_LBR1_lin/kernel`, while the
rewrite emits a top-level Dense and looks for `E-IN_LBR1_lin/kernel`.

Loading here is strict — no `by_name`, no `skip_mismatch` — so a topology
mismatch raises instead of half-loading.

In [2]:
import tensorflow as tf
for g in tf.config.experimental.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)

import msn_demo_arch as demo

AE = demo.PCT_AE_Multimodal(bert_model=demo.bert_model,
                            PCT_encoder=demo.PCT_encoder,
                            pct_decoder=demo.pct_decoder)
print(f"{AE.model.count_params()/1e6:.1f}M parameters (BERT included)")

before = [w.numpy().copy() for w in AE.model.weights[:40]]
AE.model.load_weights(os.path.join(REPO, paths.MSN_WEIGHTS))          # strict
changed = sum(1 for b, w in zip(before, AE.model.weights) if not np.array_equal(b, w.numpy()))
assert changed > 30, f"only {changed}/40 tensors changed -- wrong architecture module"
print(f"{changed}/40 leading tensors overwritten -- weights really loaded")

2026-09-08 02:29:32.923771: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-08 02:29:32.923794: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-08 02:29:32.924516: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at bert-base-uncased were not used when initializing TFBertModel: ['nsp___cls', 'mlm___cls']
- This IS expected if you are initializing TFBertModel from the check

296.9M parameters (BERT included)
40/40 leading tensors overwritten -- weights really loaded


## 3 · The comparison

Both sides were scored fold by fold, on the same 20 skulls each time — the
baseline has to sit the same exam as the model it is compared with. The baseline
additionally averages five stochastic draws per skull, because the upstream
sampler is not deterministic.

⚠️ Naming trap in `experiments_log/`: `baseline_es20` is **this project's own
first training run**, not this. It is one of the four runs invalidated by the
learning-rate patience conflict. The released weights live under
`pretrained_baseline/`.

In [3]:
import report as rp

CFG = "cd_rep05_full"
runs = rp.load_runs(REPO, [f"msn_skullfix/{CFG}_f{f}" for f in range(5)])
df = pd.read_csv(os.path.join(REPO, "experiments_log", "eval_all_runs.csv"), dtype={"id": str})
fdf = rp.fold_frame(df[df.run.isin([r.label for r in runs])], runs)

BASE_DIR = os.path.join(REPO, "experiments_log", "pretrained_baseline")
base = {f: pd.read_csv(os.path.join(BASE_DIR, f"eval_f{f}.csv"), dtype={"id": str})
        for f in range(5)}

METRIC = "defect_cov_mm"
print(f"{METRIC}, fold by fold\n")
print(f"{'fold':>5}{'baseline':>12}{CFG:>16}{'ratio':>9}")
print("-" * 42)
rows = []
for f in range(5):
    b = base[f][METRIC].mean()
    o = fdf[fdf.fold == f][METRIC].mean()
    rows.append((b, o))
    print(f"{f:>5}{b:12.3f}{o:16.3f}{b / o:9.2f}x")
b_m = np.array([r[0] for r in rows]); o_m = np.array([r[1] for r in rows])
print("-" * 42)
print(f"{'mean':>5}{b_m.mean():12.3f}{o_m.mean():16.3f}{b_m.mean() / o_m.mean():9.2f}x")
print(f"{'std':>5}{b_m.std(ddof=1):12.3f}{o_m.std(ddof=1):16.3f}")
print(f"\nSame direction in {sum(b > o for b, o in rows)}/5 folds.")

print(f"\n\nEvery metric, fold means\n")
print(f"{'':22}{'baseline':>12}{CFG:>18}{'ratio':>9}")
print("-" * 61)
for m in ("CD_t_mm", "HD95_mm", "defect_cov_mm", "defect_HD95_mm",
          "defect_prec_mm", "defect_n_pred", "defect_F1@0.05"):
    b = np.mean([base[f][m].mean() for f in range(5)])
    o = np.mean([fdf[fdf.fold == f][m].mean() for f in range(5)])
    print(f"{m:22}{b:12.3f}{o:18.3f}{b / o:9.2f}x")

defect_cov_mm, fold by fold

 fold    baseline   cd_rep05_full    ratio
------------------------------------------
    0      10.989           3.222     3.41x
    1      10.345           3.266     3.17x
    2      10.935           3.126     3.50x
    3      10.380           3.207     3.24x
    4      10.500           3.328     3.15x
------------------------------------------
 mean      10.630           3.230     3.29x
  std       0.309           0.075

Same direction in 5/5 folds.


Every metric, fold means

                          baseline     cd_rep05_full    ratio
-------------------------------------------------------------
CD_t_mm                     10.811             6.335     1.71x
HD95_mm                     12.532             5.901     2.12x
defect_cov_mm               10.630             3.230     3.29x
defect_HD95_mm              17.198             4.968     3.46x
defect_prec_mm               3.380             3.005     1.12x
defect_n_pred               32.936           34

### What this does and does not say

⭐ **The whole-cloud metric understates the gap by half.** `CD_t` puts the
baseline at roughly 1.7x, the defect region at roughly 3.3x. Most of a whole
cloud is surface the input already shows, so a model scores well there by
copying; only the defect region measures completion.

⚠️ **`defect_prec_mm` looks like a near-tie and is not one.** Read it next to
`defect_n_pred`: the baseline puts ~33 points into a hole holding 380 ground-truth
points. It barely fills the defect at all, and the few points it does place happen
to land near the surface. The metric is gameable by not predicting.

⚠️ **Do not call this zero-shot.** The released weights saw skull data in
training — MedShapeNet covers bones and draws partly on AutoImplant, which is
where SkullFix comes from. Two consequences, pointing opposite ways:

- The baseline is *not* handicapped by unfamiliarity, so the gap is attributable
  to specialisation rather than domain shift. That is the stronger claim.
- If MedShapeNet's AutoImplant subset overlaps these validation skulls, the
  baseline was scored on data it trained on, and this project's margin is
  **understated**. Worth stating in writing: the uncertainty only makes the
  conclusion more conservative.

⚠️ Finally, this is *"a general foundation model applied to skulls"* against
*"trained on skulls from scratch"* — not two methods competing at one task.

## 4 · One skull

The upstream sampler draws centroids with a stateful RNG, so the same input gives
a slightly different output each call. Measured spread across draws is 0.03 mm on
CD_t — negligible against a 4.5 mm gap, and the reason the script averages five.

In [4]:
import plotly.graph_objects as go

tok = demo.BertTokenizer.from_pretrained("bert-base-uncased")
enc = tok.encode_plus("skull", add_special_tokens=True, max_length=128,
                      padding="max_length", truncation=True, return_tensors="tf")

val = rp.Run(REPO, f"msn_skullfix/{FOLD}").meta["val_ids"]
k = int(np.where(ids == val[0])[0][0])

tf.keras.utils.set_random_seed(42)
with tf.device(DEVICE):
    pred = AE.model([tf.convert_to_tensor(inputs[k][None], tf.float32),
                     tf.zeros([1, 1, 1]), enc["input_ids"], enc["attention_mask"]],
                    training=False).numpy()[0]

fig = go.Figure()
for pts, name, colour, size in [(inputs[k], "input (defective)", "#D32F2F", 1.6),
                                (pred,      "baseline output",   "#1565C0", 1.6),
                                (gt[k],     "ground truth",      "#2E7D32", 1.2)]:
    fig.add_scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers",
                      marker=dict(size=size, color=colour, opacity=0.55), name=name)
fig.update_layout(title=f"skull {ids[k]} — released weights, no skull-specific training",
                  scene=dict(aspectmode="data"), height=650,
                  margin=dict(l=0, r=0, b=0, t=40)).show()